<a href="https://colab.research.google.com/github/SoniaCamila13/Clasificacion_correos_BERT_USFQ/blob/main/04_experimentacion_y_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Se aplica el pipe line inicial de la primera prueba con BERT

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("correos_para_limpieza.csv", sep=';')

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df["texto"] = df["Asunto"].fillna("") + " " + df["Cuerpo"].fillna("")

In [ ]:
df[["Asunto", "Cuerpo", "texto"]].head()

In [ ]:
import re

def remove_disclaimers(text):
    patterns = [
        r"este mensaje.*",
        r"este correo.*",
        r"aviso de confidencialidad.*",
        r"confidencialidad.*",
        r"this message.*",
        r"this email.*",
        r"confidential.*"
    ]

    for p in patterns:
        text = re.sub(p, "", str(text), flags=re.IGNORECASE | re.DOTALL)

    return text

In [ ]:
def remove_threads(text):
    patterns = [
        r"de:.*",
        r"from:.*",
        r"enviado:.*",
        r"sent:.*"
    ]

    for p in patterns:
        text = re.sub(p, "", str(text), flags=re.IGNORECASE | re.DOTALL)

    return text

In [ ]:
def clean_text(text):
    text = re.sub(r'\S+@\S+', '', str(text))  # eliminar correos
    text = re.sub(r'http\S+|www.\S+', '', text)  # eliminar links
    text = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚñÑ0-9\s]', '', text)  # quitar símbolos raros
    text = re.sub(r'\s+', ' ', text).strip()  # quitar espacios extra

    return text

In [ ]:
df["texto"] = df["texto"].apply(remove_disclaimers)
df["texto"] = df["texto"].apply(remove_threads)
df["texto_limpio"] = df["texto"].apply(clean_text)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label"] = le.fit_transform(df["Etiqueta"])

In [ ]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(label_mapping)

In [ ]:
df["label"].value_counts()

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["texto_limpio", "label"]])

Ahora sí, para empezar con las validaciones

PASO 1: Dividir train y test

In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)

PASO 2: Dividir train, test, validation

In [ ]:
train_val = dataset["train"].train_test_split(test_size=0.1, seed=42)

dataset["train"] = train_val["train"]
dataset["validation"] = train_val["test"]

Ahora tenemos:

*   train: entrena el modelo
*   validation: pruebas configuraciones
*   test: evaluación final (NO se toca hasta el final)






In [ ]:
dataset

El conjunto de datos fue dividido en tres subconjuntos: entrenamiento, validación y prueba. El conjunto de validación se utilizó para evaluar diferentes configuraciones del modelo, mientras que el conjunto de prueba se reservó para la evaluación final.

Ahora se definen las configuraciones:

In [ ]:
configs = [
    {"learning_rate": 2e-5, "epochs": 3},
    {"learning_rate": 3e-5, "epochs": 3},
    {"learning_rate": 2e-5, "epochs": 4}
]

Aquí se prueba:

*   qué pasa si se aumenta learning rate
*   qué pasa si se aumentan las epochs


En base a qué criterio?
- Usamos: F1-score en validation

Porque:

- hay desbalance de clases
- combina precision + recall
- es la mejor métrica para NLP

Ahora viene la tokenización:

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

def tokenize(batch):
    return tokenizer(batch["texto_limpio"], padding=True, truncation=True)

dataset = dataset.map(tokenize, batched=True)

dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Función de métricas:

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

- logits: lo que el modelo predice
- labels: la respuesta correcta

¿Qué significa cada una?
- Accuracy: % de aciertos
- Precision: qué tan exacto es
- Recall: qué tanto detecta
- F1: balance entre precision y recall

Automatización de experimentos:

Este código hace que:
- se repita en entrenamiento varias veces
- se cree un nuevo modelo por cada configuración
- se defina cómo entrenar
- entrena
- evalúa en validation
- guarda los resultados

Entones, antes se entrenaba un sólo modelo sin comparación

Ahora se entrenan varios modelos y se elige al mejor

In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

results = []

for i, config in enumerate(configs):
    print(f"\n🔹 Ejecutando configuración {i+1}: {config}")

    model = BertForSequenceClassification.from_pretrained(
        "dccuchile/bert-base-spanish-wwm-cased",
        num_labels=len(df["label"].unique())
    )

    training_args = TrainingArguments(
        output_dir=f"./results_{i}",
        learning_rate=config["learning_rate"],
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=8
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_result = trainer.evaluate()

    results.append({
        "config": config,
        "f1": eval_result["eval_f1"],
        "accuracy": eval_result["eval_accuracy"]
    })

print("\nRESULTADOS:")
for r in results:
    print(r)

VER RESULTADOS:

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df

Configuración ganadora:
- learning_rate = 2e-5
- epochs = 3

Obtuvo el mejor desempeño en validation sin incrementar complejidad ni riesgo de sobreajuste


Interpretación learning rate:
2e-5 > 3e-5
- learning rate más alto empeoró el modelo
- el modelo aprende demasiado rápido y no ajusta bien los patrones


Interpretación epochs:
- 4 epochs NO mejoró frente a 3
- puede estar empezando a sobreajustar (overfitting)

Conclusión:

- Más entrenamiento ≠ mejor modelo
- Más learning rate ≠ mejor modelo

equilibrio = clave

Se evaluaron múltiples configuraciones del modelo BERT variando la tasa de aprendizaje y el número de épocas.

Los resultados muestran que la mejor configuración corresponde a una tasa de aprendizaje de 2e-5 y 3 épocas, alcanzando un F1-score de 0.933.

Se observa que un incremento en la tasa de aprendizaje (3e-5) produce una disminución en el desempeño del modelo, lo cual sugiere un proceso de aprendizaje menos estable.

Asimismo, el aumento en el número de épocas no generó mejoras significativas, lo que indica la posible aparición de sobreajuste.

Estos resultados evidencian la importancia de ajustar cuidadosamente los hiperparámetros para optimizar el rendimiento del modelo.

Ahora se realiza el train con el modelo ganador:

In [ ]:
# =========================
# MODELO FINAL (GANADOR)
# =========================

from transformers import BertForSequenceClassification, Trainer, TrainingArguments

# 1. Crear modelo nuevo
model = BertForSequenceClassification.from_pretrained(
    "dccuchile/bert-base-spanish-wwm-cased",
    num_labels=len(df["label"].unique())
)

# 2. Configuración GANADORA
training_args = TrainingArguments(
    output_dir="./final_model",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8
)

# 3. Trainer (IMPORTANTE: usamos TEST)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],  # 🔥 evaluación final
    compute_metrics=compute_metrics
)

# 4. Entrenar
trainer.train()

# 5. Evaluación final
final_results = trainer.evaluate()

print(final_results)

Ahora ya no usamos "Validation" sino "test" para la evaluación final real

Interpretación:
- El modelo generaliza bien (no se cayó en test)
- No hay sobreajuste fuerte
- El rendimiento es estable y confiable

COMPARACIÓN FINAL

| Modelo                         | F1-score    |
| ------------------------------ | ----------- |
| Regresión Logística (baseline) | 0.85        |
| BERT (final)                   | **0.92** 🔥 |


COMPARACIÓN FINAL

El modelo final fue entrenado utilizando la mejor configuración identificada durante la fase experimental (learning rate = 2e-5, 3 épocas).

La evaluación sobre el conjunto de prueba arrojó un F1-score de 0.92 y una precisión de 0.92, evidenciando un alto desempeño del modelo en la clasificación de correos electrónicos.

Los resultados obtenidos son consistentes con los observados en el conjunto de validación, lo que indica una adecuada capacidad de generalización y ausencia de sobreajuste significativo.

En comparación con el modelo baseline basado en Regresión Logística (F1 = 0.85), el modelo BERT presenta una mejora sustancial, confirmando la efectividad de los modelos de lenguaje profundo para tareas de procesamiento de lenguaje natural.